### 234  REALHW 小红的流水线阶段划分


链接： [234  REALHW 小红的流水线阶段划分](https://www.nowcoder.com/practice/026580fd0ddc40b39e56a41b6525ce80?tpId=390&tqId=11745450&sourceUrl=%2Fexam%2Foj%2Fta%3FtpId%3D37)


![image.png](./desc.png)


In [ ]:
# 朴素 dp

import sys
INF = 1e9 + 1
line = sys.stdin.readline().split()
n, p, T = int(line[0]), int(line[1]), int(line[2])
t = [0 for _ in range(n + 2)]
c = [0 for _ in range(n + 2)]

line = sys.stdin.readline().split()
it = iter(line)
for i in range(1, n + 1):
    t[i] = int(next(it))

line = sys.stdin.readline().split()
it = iter(line)
for i in range(1, n):
    c[i] = int(next(it))

s = [0 for i in range(n + 2)]
for i in range(1, n + 1):
    s[i] = s[i - 1] + t[i]

# dp[k][i]
# 状态表示：对于前i个模型，分成k个阶段，这种分法下的最低通信开销
# 状态转移方程：dp[k][i] = min([k - 1][j] + cost(j), dp[k][i])

dp = [[INF for i in range(n + 1)] for k in range(p + 1)]

dp[0][0] = 0 # dp[0][i > 0] 都不可达，因为使用i个模型，却只有0层，明显是不可能的

for k in range(1, p + 1):
    q = []
    for i in range(1, n + 1):
        # for j in range(1, i): # 漏了 j = 0
        for j in range(0, i):
            # 先看 i 能否能和 j + 1 同阶段
            st = s[i] - s[j] # 从 j + 1, j + 2, ... , i 的总时间开销
            if st <= T:
                dp[k][i] = min(dp[k][i], dp[k - 1][j] + c[j])
            # # 再记录 i 单独一个阶段
            # dp[k][i] = min(dp[k][i], dp[k - 1][i - 1] + c[i - 1])

ans = dp[p][n]
if ans == INF:
    ans = -1
print(int(ans))


#### 单调队列优化的原理：

核心观察：

1. 层时间是正数，所以前缀和 s 递增。

2. 对于每个 i，合法的 j 是一个连续区间 [L, i-1]。

3. 随着 i 变大，L 只会往右移，不会往左移。

所以 j 的候选区间就像一个滑动的窗口，从左到右移动。

我们要的： 窗口里 dp[k-1][j] + 切点开销 最小的那个 j。

单调队列就是干这个的： 用一个队列维护窗口，队头永远是最小值的 j。

In [ ]:
# 单调队列优化

import sys
INF = int(1e9 + 1)
line = sys.stdin.readline().split()
n, p, T = int(line[0]), int(line[1]), int(line[2])
t = [0 for _ in range(n + 2)]
c = [0 for _ in range(n + 2)]

line = sys.stdin.readline().split()
it = iter(line)
for i in range(1, n + 1):
    t[i] = int(next(it))

line = sys.stdin.readline().split()
it = iter(line)
for i in range(1, n):
    c[i] = int(next(it))

s = [0 for i in range(n + 2)]
for i in range(1, n + 1):
    s[i] = s[i - 1] + t[i]

# dp[k][i]
# 状态表示：对于前i个模型，分成k个阶段，这种分法下的最低通信开销
# 状态转移方程：dp[k][i] = min([k - 1][j] + cost(j), dp[k][i])

dp = [[INF for i in range(n + 1)] for k in range(p + 1)]

dp[0][0] = 0 # dp[0][i > 0] 都不可达，因为使用i个模型，却只有0层，明显是不可能的

prev = [INF] * (n + 1)
prev[0] = 0 # 对于第0段的第0个模型，无需任何开销

for k in range(1, p + 1):
    curr = [INF] * (n + 1)
    q = []
    for i in range(1, n + 1):
        j_new = i - 1
        v_new = prev[j_new] + c[j_new]
        # 维护单调队列
        while q and prev[q[-1]] + c[q[-1]] >= v_new:
            q.pop() # 弹出当前队尾
            
        q.append(j_new)
        
        # 同时需要用T这个条件进行维护
        while q and s[i] - s[q[0]] >= T:
            q.pop(0)
        
        # 更新
        if q:
            # 取出 dp[k-1][j] + cost[j] 最小的j
            j = q[0]
            # 计算
            v = prev[j] + c[j]
            # 更新
            curr[i] = min(curr[i], v)
    
    # 循环完当前阶段，所有模型后，开始下一阶段
    prev = curr

# 所有阶段循环完毕，最后的 curr 存在 prev 中
ans = prev[n] if prev[n] != INF else -1
print(int(ans))

```
您的代码已保存
通过全部用例
运行时间
1125ms
占用内存
12684KB
```
